In [ ]:
from pypdf import PdfReader
import os
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter
)
import chromadb
from langchain_google_genai import ChatGoogleGenerativeAI
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [ ]:
os.environ["GOOGLE_API_KEY"] = "API_KEY" #We are Using GEMINI MODELS

In [ ]:
client = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
def word_wrap(text, width=87):
    """
    Wraps the given text to the specified width.

    Args:
    text (str): The text to wrap.
    width (int): The width to wrap the text to.

    Returns:
    str: The wrapped text.
    """
    return "\n".join([text[i : i + width] for i in range(0, len(text), width)])

In [ ]:
read=PdfReader("2024_Annual_Report.pdf")

In [ ]:
# print(read.pages[1].extract_text())

In [ ]:
pdf_txt = [p.extract_text().strip() for p in read.pages]

In [ ]:
# pdf_txt

In [ ]:
pdf_txt = [text for text in pdf_txt if text]

In [ ]:
#pdf_txt (Number of pages of docs)

In [ ]:
char_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
    chunk_size=1000,
    chunk_overlap=0,
    length_function=len,
)

In [ ]:
char_split_text = char_splitter.split_text('\n\n'.join(pdf_txt))

In [ ]:
len(char_split_text)

In [ ]:
token_split = SentenceTransformersTokenTextSplitter(
    chunk_overlap=0,
    tokens_per_chunk=256
)

In [ ]:
token_split_txt=[]
for txt in char_split_text:
  token_split_txt+=token_split.split_text(txt)

In [ ]:
emb_func = SentenceTransformerEmbeddingFunction()

In [ ]:
chroma_cli = chromadb.Client()
chroma_collection = chroma_cli.create_collection(name="annual_report",embedding_function=emb_func)

In [ ]:
ids=[str(i) for i in range(len(token_split_txt))]

In [ ]:
chroma_collection.add(
    documents=token_split_txt,
    ids=ids
)

In [ ]:
query = "what was the total revenue for the year"

In [ ]:
res = chroma_collection.query(query_texts=query, n_results=3)

In [ ]:
print(res['documents'][0][0])